# xCounter — Identificação Simples por `play_pattern_name`

**Disciplina:** COE609 — Ciência de Dados Aplicada ao Futebol (UFRJ, 2026.1)

## Objetivo

Versão **simplificada** do pipeline xCounter. Escolhas que tornam este notebook
mais enxuto que o original:

1. **Identificação** dos contra-ataques via rótulo `play_pattern_name == 'From Counter'`.
2. **Features espaciais** do freeze frame 360 (iguais às do notebook original).
3. **Integração com o FM2023** por *match fuzzy* (similaridade) + aliases manuais.

**Regra de cadeia:** uma cadeia *inicia* quando uma linha tem
`play_pattern_name == 'From Counter'` e *termina* quando a coluna deixa de ser
`'From Counter'`. Linhas `From Counter` consecutivas formam a mesma cadeia.

## Partida analisada

Copa do Mundo FIFA 2022 — **País de Gales × Irã** (`match_id = 3857273`).

In [ ]:
import os
# Executa a partir da raiz do projeto (este notebook fica em notebooks/)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Diretório de trabalho:', os.getcwd())

In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import os
import re
import zipfile
import numpy as np
import pandas as pd
from difflib import SequenceMatcher

from mplsoccer import Sbopen
from unidecode import unidecode

print('Bibliotecas carregadas com sucesso.')
print(f'  pandas {pd.__version__} | numpy {np.__version__}')

Bibliotecas carregadas com sucesso.
  pandas 2.0.3 | numpy 1.24.3


## Parâmetros

O critério de sucesso é mantido igual ao do notebook original para comparabilidade.

In [2]:
# ==================
# PARÂMETROS GLOBAIS
# ==================
MATCH_ID       = 3857273   # Copa do Mundo 2022: País de Gales × Irã
COMPETITION_ID = 43        # FIFA World Cup
SEASON_ID      = 106       # 2022

GOAL_X = 120.0  # Coordenada x do gol adversário (sistema StatsBomb: 0-120)
GOAL_Y = 40.0   # Coordenada y do centro do gol

PLAY_PATTERN_ALVO = 'From Counter'  # rótulo StatsBomb que define a cadeia

# Atributos físicos do FM2023 a incorporar
FM_ATTRS    = ['Pac', 'Acc', 'Sta', 'Str', 'Agi', 'Bal', 'Jum']
FM_CSV_PATH = 'data/fm2023.csv'
FM_ZIP_PATH = 'data/fm2023.zip'
LIMIAR_FUZZY = 0.60  # similaridade mínima para aceitar um match no FM

# --- Critério de sucesso da cadeia ---
# 'qualquer_chute' : sucesso se a cadeia contém qualquer Shot
# 'xg_threshold'   : sucesso se contém Shot com shot_statsbomb_xg >= xg_threshold
# 'gol'            : sucesso somente se algum Shot da cadeia resultou em gol
config = {
    'criterio_sucesso': 'xg_threshold',
    'xg_threshold':     0.10,
}

print('Parâmetros configurados.')
print(f'  Match ID: {MATCH_ID} | play_pattern alvo: {PLAY_PATTERN_ALVO!r}')
print(f"  Critério de sucesso: {config['criterio_sucesso']} (xG >= {config['xg_threshold']})")

Parâmetros configurados.
  Match ID: 3857273 | play_pattern alvo: 'From Counter'
  Critério de sucesso: xg_threshold (xG >= 0.1)


## Etapa 1 — Carregamento dos Dados

Carregamos eventos, **táticas** (defensores titulares na Etapa 4) e os **freeze
frames 360** (features espaciais na Etapa 3) via `mplsoccer.Sbopen`.

In [3]:
print('=== Etapa 1: Carregamento dos Dados ===\n')
parser = Sbopen()

# Eventos + táticas (1o e 4o DataFrames retornados por .event)
df_event, _, _, df_tactics = parser.event(MATCH_ID)
# Freeze frames 360 (posições por evento)
df_360, _ = parser.frame(MATCH_ID)

print(f'Eventos carregados: {len(df_event)} | linhas 360: {len(df_360)}')
print('\nDistribuição de play_pattern_name:')
print(df_event['play_pattern_name'].value_counts(dropna=False))

=== Etapa 1: Carregamento dos Dados ===

Eventos carregados: 3133 | linhas 360: 42319

Distribuição de play_pattern_name:
play_pattern_name
Regular Play      1184
From Throw In      898
From Free Kick     394
From Goal Kick     251
From Keeper        213
From Corner        111
From Counter        51
From Kick Off       30
Other                1
Name: count, dtype: int64


## Etapa 2 — Identificação de Contra-Ataques por `play_pattern_name`

Agrupa **corridas consecutivas** de eventos com `play_pattern_name == 'From Counter'`
via `(mask != mask.shift()).cumsum()`. Para cada cadeia calcula métricas e o rótulo
de sucesso (qualquer `Shot` na cadeia; equipe = a do primeiro evento).

In [4]:
# ============================================================
# ETAPA 2: IDENTIFICAÇÃO SIMPLES VIA play_pattern_name
# ============================================================

def identificar_contra_ataques_simples(df_event, cfg):
    """
    Identifica contra-ataques agrupando corridas consecutivas de eventos com
    play_pattern_name == 'From Counter'.
    """
    df = df_event.sort_values(['period', 'minute', 'second', 'index']).reset_index(drop=True)

    eh_counter = df['play_pattern_name'] == PLAY_PATTERN_ALVO
    grupo = (eh_counter != eh_counter.shift()).cumsum()

    criterio = cfg['criterio_sucesso']
    xg_thr   = cfg.get('xg_threshold', 0.10)

    contra_ataques = []
    for _, cadeia in df[eh_counter].groupby(grupo):
        primeiro = cadeia.iloc[0]
        ultimo   = cadeia.iloc[-1]

        x_validos = cadeia['x'].dropna()
        x_ini = float(x_validos.iloc[0]) if len(x_validos) else np.nan
        x_fim = ultimo['end_x']
        if pd.isna(x_fim):
            x_fim = ultimo['x']
        x_fim = float(x_fim) if pd.notna(x_fim) else x_ini
        prog  = (x_fim - x_ini) if pd.notna(x_ini) else np.nan

        t0 = primeiro['minute'] * 60 + primeiro['second']
        t1 = ultimo['minute'] * 60 + ultimo['second']

        chutes = cadeia[cadeia['type_name'] == 'Shot']
        terminou_chute = len(chutes) > 0
        xg_final = float(chutes['shot_statsbomb_xg'].fillna(0).max()) if terminou_chute else 0.0

        if criterio == 'qualquer_chute':
            sucesso = terminou_chute
        elif criterio == 'gol':
            sucesso = bool((chutes['outcome_name'].astype(str).str.lower() == 'goal').any())
        else:  # xg_threshold
            sucesso = bool(terminou_chute and xg_final >= xg_thr)

        contra_ataques.append({
            'id_inicio':      primeiro['id'],
            'player_name':    str(primeiro.get('player_name', '')),
            'team':           primeiro['team_name'],
            'period':         int(primeiro['period']),
            'minute':         int(primeiro['minute']),
            'second':         int(primeiro['second']),
            'tipo_inicio':    primeiro['type_name'],
            'num_eventos':    len(cadeia),
            'x_inicio':       x_ini,
            'x_fim':          x_fim,
            'progressao_x':   prog,
            'duracao_s':      float(t1 - t0),
            'terminou_chute': terminou_chute,
            'xg_final':       xg_final,
            'sucesso':        sucesso,
        })

    return contra_ataques


print('Função identificar_contra_ataques_simples definida.')

Função identificar_contra_ataques_simples definida.


In [5]:
contra_ataques = identificar_contra_ataques_simples(df_event, config)
df_ca = pd.DataFrame(contra_ataques)

print(f'Contra-ataques identificados: {len(df_ca)}')
if len(df_ca) > 0:
    print(f"Taxa de sucesso geral:        {df_ca['sucesso'].mean():.1%}\n")
    cols = ['team', 'minute', 'second', 'tipo_inicio', 'num_eventos',
            'progressao_x', 'duracao_s', 'terminou_chute', 'xg_final', 'sucesso']
    display(df_ca[cols])

Contra-ataques identificados: 7
Taxa de sucesso geral:        42.9%



,team,minute,second,tipo_inicio,num_eventos,progressao_x,duracao_s,terminou_chute,xg_final,sucesso
0,Iran,19,0,Pass,9,69.8,10.0,True,0.017789,False
1,Iran,21,32,Interception,5,33.6,4.0,False,0.000000,False
2,Iran,50,48,Duel,10,92.7,10.0,True,0.287996,True
3,Wales,51,9,Goal Keeper,3,113.5,0.0,True,0.214015,True
4,Iran,63,30,Ball Recovery,8,62.7,9.0,True,0.075856,False
5,Iran,83,37,Pass,10,64.8,6.0,False,0.000000,False
6,Iran,100,27,Ball Recovery,6,73.3,10.0,True,0.199331,True


## Resumo por Equipe

In [6]:
if len(df_ca) > 0:
    resumo = df_ca.groupby('team').agg(
        n_ca=('sucesso', 'count'),
        com_sucesso=('sucesso', 'sum'),
        taxa=('sucesso', 'mean'),
        prog_media=('progressao_x', 'mean'),
        duracao_media=('duracao_s', 'mean'),
    )
    resumo['taxa'] = (resumo['taxa'] * 100).round(1)
    print('Resumo por equipe:')
    display(resumo)
else:
    print('Nenhum contra-ataque identificado.')

Resumo por equipe:


,n_ca,com_sucesso,taxa,prog_media,duracao_media
team,,,,,
Iran,6,2,33.3,66.15,8.166667
Wales,1,1,100.0,113.50,0.000000


## Etapa 3 — Features Espaciais do Freeze Frame 360

Para cada contra-ataque, lê o freeze frame 360 do **evento de início** e calcula as
mesmas 7 features do notebook original. Companheiros (`teammate=True`) são os
atacantes do contra-ataque; adversários (`teammate=False`) são os defensores.

| Feature | Cálculo |
|---------|---------|
| `dist_gol_defensor_proximo` | dist. gol → defensor mais avançado (maior `x`) |
| `dist_gol_atacante_avancado` | dist. gol → atacante mais avançado |
| `diferencial_profundidade` | `x_atacante_avancado − x_defensor_mais_avancado` |
| `num_defensores_entre_bola_gol` | defensores com `x > x_bola` |
| `largura_linha_defensiva` | espalhamento em `y` dos 3 defensores mais avançados |
| `dist_bola_gol` | dist. da bola ao gol |
| `superioridade_terco_ofensivo` | nº atacantes − nº defensores com `x > 80` |

> Cadeias sem freeze frame disponível ficam com as features espaciais em `NaN`
> (mantemos a linha, em vez de descartá-la).

In [7]:
# ============================================================
# ETAPA 3: FEATURES ESPACIAIS DO FREEZE FRAME 360
# ============================================================

def dist_eucl(x, y, x0=GOAL_X, y0=GOAL_Y):
    """Distância Euclidiana de (x, y) ao gol adversário (x=120, y=40)."""
    return float(np.sqrt((x - x0) ** 2 + (y - y0) ** 2))


def extrair_features_espaciais(event_id, df_event, df_360):
    """Extrai 7 features espaciais do freeze frame 360 ou None se indisponível."""
    ev = df_event[df_event['id'] == event_id]
    if ev.empty:
        return None
    ev = ev.iloc[0]

    frame = df_360[df_360['id'] == event_id]
    if frame.empty:
        return None

    bola_x, bola_y = ev.get('x'), ev.get('y')
    if bola_x is None or pd.isna(bola_x):
        return None
    bola_x, bola_y = float(bola_x), float(bola_y)

    tm = frame[frame['teammate'] == True].copy()   # companheiros (atacantes)
    op = frame[frame['teammate'] == False].copy()  # adversários (defensores)
    if len(op) < 1 or len(tm) < 1:
        return None

    x_def_av  = float(op['x'].max())
    y_def_av  = float(op.loc[op['x'].idxmax(), 'y'])
    x_atac_av = float(tm['x'].max())
    y_atac_av = float(tm.loc[tm['x'].idxmax(), 'y'])

    n_def_entre = int((op['x'] > bola_x).sum())
    top3 = op.nlargest(min(3, len(op)), 'x')
    largura = float(top3['y'].max() - top3['y'].min()) if len(top3) >= 2 else 0.0
    sup_of  = int((tm['x'] > 80).sum()) - int((op['x'] > 80).sum())

    return {
        'id_inicio':                     event_id,
        'dist_gol_defensor_proximo':     dist_eucl(x_def_av, y_def_av),
        'dist_gol_atacante_avancado':    dist_eucl(x_atac_av, y_atac_av),
        'diferencial_profundidade':      x_atac_av - x_def_av,
        'num_defensores_entre_bola_gol': n_def_entre,
        'largura_linha_defensiva':       largura,
        'dist_bola_gol':                 dist_eucl(bola_x, bola_y),
        'superioridade_terco_ofensivo':  sup_of,
    }


feat_rows, sem_frame = [], 0
for ca in contra_ataques:
    feats = extrair_features_espaciais(ca['id_inicio'], df_event, df_360)
    if feats is not None:
        feat_rows.append(feats)
    else:
        sem_frame += 1

df_feats_esp = pd.DataFrame(feat_rows)
print(f'Cadeias com freeze frame: {len(df_feats_esp)}/{len(contra_ataques)} (sem frame: {sem_frame})')
if len(df_feats_esp) > 0:
    display(df_feats_esp.describe().round(2))

Cadeias com freeze frame: 6/7 (sem frame: 1)


,dist_gol_defensor_proximo,dist_gol_atacante_avancado,diferencial_profundidade,num_defensores_entre_bola_gol,largura_linha_defensiva,dist_bola_gol,superioridade_terco_ofensivo
count,6.00,6.00,6.00,6.00,6.00,6.00,6.0
mean,75.93,79.00,-2.84,2.50,27.44,87.33,0.0
std,13.34,10.99,5.08,1.38,15.50,9.94,0.0
min,58.77,64.11,-10.20,1.00,10.49,77.40,0.0
25%,67.11,70.83,-5.36,2.00,19.95,78.71,0.0
50%,75.79,82.21,-3.40,2.00,25.07,86.65,0.0
75%,83.94,84.79,1.19,2.75,28.61,94.28,0.0
max,94.37,92.81,3.23,5.00,55.99,100.39,0.0


## Etapa 4 — Integração com o Football Manager 2023 (match *fuzzy* + aliases)

Hipótese central: substituir features físicas de tracking por um *prior* estático do
FM2023 (`Pac, Acc, Sta, Str, Agi, Bal, Jum`).

**Match:** (1) *aliases manuais* por UID para falsos negativos conhecidos; (2) caso
contrário, *fuzzy* (maior `SequenceMatcher.ratio`) dentro do pool da nacionalidade
(time = nacionalidade na Copa), rejeitando matches abaixo de `LIMIAR_FUZZY`.

Para cada contra-ataque: **atacante** (match individual) vs. **defensor** (média dos
defensores titulares da equipe adversária — proxy fixo por equipe).

In [8]:
# ============================================================
# ETAPA 4A: CARREGAR FM2023 + MAPA DE NACIONALIDADE
# ============================================================

def norm_nome(nome):
    """Normaliza nome: unidecode + minúsculas + remove não-letras e sufixos."""
    nome = unidecode(str(nome)).lower()
    nome = re.sub(r'[^a-z\s]', '', nome)
    nome = re.sub(r'\b(jr|sr|ii|iii|iv)\b', '', nome)
    return re.sub(r'\s+', ' ', nome).strip()


# StatsBomb country_name → código FM2023 (Nat). 32 seleções da Copa 2022.
COUNTRY_TO_FM = {
    'Argentina': 'ARG', 'Australia': 'AUS', 'Belgium': 'BEL', 'Brazil': 'BRA',
    'Cameroon': 'CMR', 'Canada': 'CAN', 'Costa Rica': 'CRC', 'Croatia': 'CRO',
    'Denmark': 'DEN', 'Ecuador': 'ECU', 'England': 'ENG', 'France': 'FRA',
    'Germany': 'GER', 'Ghana': 'GHA', 'Iran, Islamic Republic of': 'IRN',
    'Japan': 'JPN', 'Korea\xa0(South)': 'KOR', 'Mexico': 'MEX', 'Morocco': 'MAR',
    'Netherlands': 'NED', 'Poland': 'POL', 'Portugal': 'POR', 'Qatar': 'QAT',
    'Saudi Arabia': 'KSA', 'Senegal': 'SEN', 'Serbia': 'SRB', 'Spain': 'ESP',
    'Switzerland': 'SUI', 'Tunisia': 'TUN', 'United States of America': 'USA',
    'Uruguay': 'URU', 'Wales': 'WAL',
}

# Extrai o CSV do zip na primeira execução, se necessário.
if not os.path.exists(FM_CSV_PATH) and os.path.exists(FM_ZIP_PATH):
    zipfile.ZipFile(FM_ZIP_PATH).extractall('.')
    print(f'{FM_CSV_PATH} extraído de {FM_ZIP_PATH}.')

df_fm = pd.read_csv(FM_CSV_PATH, encoding='utf-8')
df_fm.columns = [c.strip() for c in df_fm.columns]
for col in df_fm.select_dtypes('object').columns:
    df_fm[col] = df_fm[col].astype(str).str.strip()
for attr in FM_ATTRS:
    df_fm[attr] = pd.to_numeric(df_fm[attr], errors='coerce')
df_fm['name_norm'] = df_fm['Name'].apply(norm_nome)
print(f'FM2023 carregado: {len(df_fm):,} jogadores.')

# Mapa time → código FM (time = nacionalidade na Copa do Mundo)
df_lineup = parser.lineup(MATCH_ID)
TEAM_TO_FM = {
    team: COUNTRY_TO_FM.get(grp['country_name'].iloc[0])
    for team, grp in df_lineup.groupby('team_name')
}
print('Mapa time → FM:', TEAM_TO_FM)

FM2023 carregado: 189,345 jogadores.
Mapa time → FM: {'Iran': 'IRN', 'Wales': 'WAL'}


In [9]:
# ============================================================
# ETAPA 4B: MATCH FUZZY (aliases por UID + similaridade)
# ============================================================

# Aliases manuais para falsos negativos do fuzzy (jogador EXISTE no FM, mas o nome
# civil completo da StatsBomb não bate com o nome curto do FM). Nome normalizado
# StatsBomb -> UID exato no FM2023 (evita homônimos, ex.: dois 'João Mário').
ALIASES_FM = {
    'lionel andres messi cuccittini':    7458500,   # Lionel Messi (ARG)
    'joao mario naval da costa eduardo':  55022391,  # João Mário (POR, 1993)
    'otavio edmilson da silva monteiro':  19186901,  # Otávio (POR)
    'ngoran suiru fai collins':           13112672,  # Collins Fai (CMR)
    'firas tariq nasser al albirakan':    23414228,  # Firas Al-Buraikan (KSA)
}


def match_fuzzy(player_name, nat_fm, df_fm, limiar=LIMIAR_FUZZY):
    """
    Casa um nome StatsBomb com o FM2023: (1) alias por UID; (2) fuzzy no pool da
    nacionalidade. Retorna (dict_atributos, nome_encontrado, similaridade) ou
    (None, player_name, similaridade) se abaixo do limiar.
    """
    alvo = norm_nome(player_name)
    if not alvo:
        return None, player_name, 0.0

    uid = ALIASES_FM.get(alvo)
    if uid is not None:
        cand = df_fm[df_fm['UID'] == uid]
        if not cand.empty:
            row = cand.iloc[0]
            return {a: row.get(a, np.nan) for a in FM_ATTRS}, row['Name'], 1.0

    pool = df_fm[df_fm['Nat'] == nat_fm] if nat_fm else df_fm
    if pool.empty:
        pool = df_fm

    sims = pool['name_norm'].apply(lambda n: SequenceMatcher(None, alvo, n).ratio())
    idx, melhor = sims.idxmax(), float(sims.max())
    if melhor < limiar:
        return None, player_name, round(melhor, 3)

    row = pool.loc[idx]
    return {a: row.get(a, np.nan) for a in FM_ATTRS}, row['Name'], round(melhor, 3)


print('Função match_fuzzy definida (com aliases manuais).')

Função match_fuzzy definida (com aliases manuais).


In [10]:
# ============================================================
# ETAPA 4C: ATRIBUTOS MÉDIOS DOS DEFENSORES TITULARES
# ============================================================
POSICOES_DEFENSIVAS = [
    'Centre Back', 'Center Back', 'Left Back', 'Right Back',
    'Left Centre Back', 'Left Center Back', 'Right Centre Back', 'Right Center Back',
    'Left Wing Back', 'Right Wing Back', 'Sweeper',
]


def get_defender_attrs_medios(df_event, df_tactics, df_fm):
    xi_ev = (df_event[df_event['type_name'] == 'Starting XI']
             [['id', 'team_name']].drop_duplicates())
    tact  = df_tactics.merge(xi_ev, on='id', how='inner')

    resultado = {}
    for team, grp in tact.groupby('team_name'):
        defs = grp[grp['position_name'].isin(POSICOES_DEFENSIVAS)]
        if defs.empty:
            defs = grp[grp['position_name'] != 'Goalkeeper']
        nat = TEAM_TO_FM.get(team)

        linhas, nomes_ok = [], []
        for _, pl in defs.iterrows():
            attrs, nome, sim = match_fuzzy(pl['player_name'], nat, df_fm)
            if attrs is not None:
                linhas.append(attrs)
                nomes_ok.append(f"{pl['player_name']} -> {nome} (sim={sim})")

        if linhas:
            dd = pd.DataFrame(linhas)
            medias = {a: float(dd[a].mean()) for a in FM_ATTRS}
        else:
            medias = {a: np.nan for a in FM_ATTRS}
        medias['_n_defensores']  = len(defs)
        medias['_n_encontrados'] = len(linhas)
        medias['_nomes_ok']      = nomes_ok
        resultado[team] = medias
    return resultado


defender_attrs = get_defender_attrs_medios(df_event, df_tactics, df_fm)
for team, info in defender_attrs.items():
    print(f"{team}: {info['_n_encontrados']}/{info['_n_defensores']} defensores no FM"
          f"  | Pac={info['Pac']:.1f} Acc={info['Acc']:.1f}")
    for nm in info['_nomes_ok']:
        print(f'    {nm}')

Iran: 4/4 defensores no FM  | Pac=11.8 Acc=12.2
    Ramin Rezaeian -> Ramin Rezaeian (sim=1.0)
    Seyed Majid Hosseini -> Seyed Mehdi Hosseini (sim=0.85)
    Morteza Pouraliganji -> Morteza Pouraliganji (sim=1.0)
    Milad Mohammadi -> Milad Mohammadi (sim=1.0)
Wales: 5/5 defensores no FM  | Pac=13.0 Acc=12.6
    Chris Mepham -> Chris Mepham (sim=1.0)
    Joe Rodon -> Joe Rodon (sim=1.0)
    Ben Davies -> Ben Davies (sim=1.0)
    Connor Roberts -> Connor Roberts (sim=1.0)
    Neco Williams -> Neco Williams (sim=1.0)


In [11]:
# ============================================================
# ETAPA 4D: CRUZAMENTO POR CONTRA-ATAQUE → DATASET FINAL
# ============================================================
equipes = list(defender_attrs.keys())
fm_rows = []

for _, ca in df_ca.iterrows():
    equipe_atac = ca['team']
    equipe_def  = next((e for e in equipes if e != equipe_atac), equipe_atac)

    nat = TEAM_TO_FM.get(equipe_atac)
    atac_attrs, atac_nome, atac_sim = match_fuzzy(ca['player_name'], nat, df_fm)
    atac_ok = atac_attrs is not None
    if not atac_ok:
        atac_attrs = {a: np.nan for a in FM_ATTRS}

    def_info  = defender_attrs.get(equipe_def, {})
    pac_atac, acc_atac = atac_attrs.get('Pac', np.nan), atac_attrs.get('Acc', np.nan)
    pac_def,  acc_def  = def_info.get('Pac', np.nan),   def_info.get('Acc', np.nan)

    def _dif(a, b):
        return (a - b) if not (pd.isna(a) or pd.isna(b)) else np.nan

    fm_rows.append({
        'id_inicio':                ca['id_inicio'],
        'atac_fm_nome':             atac_nome,
        'atac_fm_sim':              atac_sim,
        'atac_match_ok':            atac_ok,
        'pace_atacante':            pac_atac,
        'acceleration_atacante':    acc_atac,
        'stamina_atacante':         atac_attrs.get('Sta', np.nan),
        'pace_defensor':            pac_def,
        'acceleration_defensor':    acc_def,
        'diferencial_pace':         _dif(pac_atac, pac_def),
        'diferencial_acceleration': _dif(acc_atac, acc_def),
    })

df_fm_feats = pd.DataFrame(fm_rows)

# Dataset final = metadados + features espaciais (Etapa 3) + features FM (Etapa 4)
df_dataset_final = (df_ca
    .merge(df_feats_esp, on='id_inicio', how='left')
    .merge(df_fm_feats,  on='id_inicio', how='left'))

print(f'Dataset final: {len(df_dataset_final)} linhas × {len(df_dataset_final.columns)} colunas\n')
cols_show = ['team', 'minute', 'player_name', 'atac_fm_sim',
             'diferencial_profundidade', 'num_defensores_entre_bola_gol',
             'pace_atacante', 'pace_defensor', 'diferencial_pace', 'sucesso']
display(df_dataset_final[cols_show])

Dataset final: 7 linhas × 32 colunas



,team,minute,player_name,atac_fm_sim,diferencial_profundidade,num_defensores_entre_bola_gol,pace_atacante,pace_defensor,diferencial_pace,sucesso
0,Iran,19,Sardar Azmoun,1.000,3.230439,3.0,16,13.00,3.00,False
1,Iran,21,Ramin Rezaeian,1.000,-4.999481,5.0,13,13.00,0.00,False
2,Iran,50,Ehsan Hajsafi,1.000,-1.801973,2.0,11,13.00,-2.00,True
3,Wales,51,Wayne Hennessey,1.000,NaN,NaN,11,11.75,-0.75,True
4,Iran,63,Saeid Ezatolahi Afagh,0.833,-5.473972,2.0,12,13.00,-1.00,False
5,Iran,83,Alireza Jahanbakhsh,1.000,2.189655,2.0,12,13.00,-1.00,False
6,Iran,100,Mehdi Taremi,1.000,-10.197388,1.0,13,13.00,0.00,True


### Features disponíveis para modelagem

**Alvo:** `sucesso`.

- **Espaciais (Etapa 3):** `dist_gol_defensor_proximo`, `dist_gol_atacante_avancado`,
  `diferencial_profundidade`, `num_defensores_entre_bola_gol`, `largura_linha_defensiva`,
  `dist_bola_gol`, `superioridade_terco_ofensivo`.
- **Físicas FM (Etapa 4):** `pace_atacante`, `acceleration_atacante`, `stamina_atacante`,
  `pace_defensor`, `acceleration_defensor`, `diferencial_pace`, `diferencial_acceleration`.

> ⚠️ **Não usar `xg_final` como feature** — ele define o rótulo `sucesso` (vazamento de alvo).

### Limitações
- Defensor = proxy médio fixo por equipe (o freeze frame 360 não traz `player_id`).
- Sem desambiguação automática de homônimos; o `LIMIAR_FUZZY` rejeita matches fracos
  e os aliases corrigem casos conhecidos.
- Cadeias sem freeze frame ficam com features espaciais `NaN`.